## **Evaluation Metrics For Classification Models**

When evaluating a **classification** model, the following metrics are typically used: <br>
[Read More in **Confution Matrix**](#understanding-confusion-matrix-in-machine-learning)

### 1. **Accuracy**
   - The percentage of correctly classified instances out of all instances.

### 2. **Precision**

- The percentage of true positives out of all instances predicted as positive.
$$
\text{Precision} = \frac{TP}{TP + FP}
$$

### 3. **Recall (Sensitivity)**

   - The percentage of true positives out of all actual positives.

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

### 4. **F1 Score**

   - The harmonic mean of precision and recall.

$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

### 5. **AUC-ROC**

   - **Area Under the Receiver Operating Characteristic Curve (AUC-ROC)** is a performance measurement for classification problems at `various thresholds` settings.

ROC plots TPR (=Recall) vs. FPR $=\dfrac{FP}{FP+TN}$ across all decision
thresholds. **AUC has a precise probabilistic interpretation**:
$$\text{AUC} = P(\text{score}(x^+) > \text{score}(x^-))$$
i.e., the probability that a randomly chosen positive example is ranked
higher than a randomly chosen negative example by the model. This is
**exactly the Mann-Whitney U statistic**, normalized:
$$\text{AUC} = \frac{U}{n^+n^-}, \qquad U = \sum_{i\in\text{pos}}\sum_{j\in\text{neg}} \mathbb 1[\text{score}(x_i)>\text{score}(x_j)]$$
This is why AUC=0.5 means "no better than random ranking" (a random
score is equally likely to rank the positive above or below the
negative) and AUC=1.0 means perfect ranking (every positive outranks
every negative).

In [2]:
# AUC via Mann-Whitney U
pos = [0.9, 0.4]; neg = [0.3, 0.6]
U = sum(p > n for p in pos for n in neg)
print(U/(len(pos)*len(neg)))   # 0.75

from sklearn.metrics import roc_auc_score
y_true = [1,1,0,0]; scores = [0.9,0.4,0.3,0.6]
print(roc_auc_score(y_true, scores))   # 0.75 -- matches

0.75
0.75


### **6. Statistical Significance - Paired t-test for comparing two models**

Given per-fold (or per-sample) error differences $d_i = \text{err}_A^{(i)}-\text{err}_B^{(i)}$
across $k$ folds, test $H_0:\ E[d]=0$ via:
$$t = \frac{\bar d}{s_d/\sqrt k}, \qquad s_d = \text{sample std of } d_i$$
compared against a $t$-distribution with $k-1$ degrees of freedom. This
formalizes the standard-error reasoning already introduced in
`05_Model_Evaluation\05_cross_validation.ipynb`  answering "is Model A's better CV
score than Model B's likely real, or within noise?" rigorously rather than
just eyeballing the two numbers.

#### Worked numerical example

5-fold errors: 

Model A = {0.10,0.12,0.09,0.11,0.13}, 

Model B = {0.15,0.14,0.16,0.13,0.17}. 

Differences $d=\{-0.05,-0.02,-0.07,-0.02,-0.04\}$,


$\bar d=-0.04$. Sample variance: deviations from $\bar d$ are
$\{-0.01,0.02,-0.03,0.02,0.00\}$, squared sum $=0.0018$, sample variance
(dividing by $k-1=4$) $=0.00045$, so $s_d\approx0.0212$.
$$t = \frac{-0.04}{0.0212/\sqrt5} = \frac{-0.04}{0.00949}\approx-4.22$$
With 4 degrees of freedom, $|t|=4.22$ exceeds the critical value
(~2.78 at $\alpha=0.05$ two-tailed) → **reject $H_0$**: Model A is
significantly better, not just luckier on this particular CV split.

In [1]:
# Paired t-test

import numpy as np
from scipy import stats

errA = np.array([0.10,0.12,0.09,0.11,0.13])
errB = np.array([0.15,0.14,0.16,0.13,0.17])
t_stat, p_val = stats.ttest_rel(errA, errB)
print(t_stat, p_val)   # ~-4.66, p<0.05

-4.216370213557841 0.013516881521817502


##
---

## **Understanding Confusion Matrix in Machine Learning**

A **Confusion Matrix** is a fundamental tool used to evaluate the performance of a classification algorithm in machine learning. It allows us to visualize the performance of a model, particularly in terms of its **accuracy** and **errors**. The matrix compares the predicted labels from the model with the true labels, helping us understand where the model is making mistakes.

### What is a Confusion Matrix?

A confusion matrix is a table that is used to describe the performance of a classification algorithm. It compares the predicted classifications with the actual classifications. The matrix is typically structured as follows:

|               | Predicted Positive | Predicted Negative |
|---------------|--------------------|--------------------|
| **Actual Positive** | True Positive (TP)  | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN)  |

Here’s a breakdown of the terms:

1. **True Positive (TP):** These are the cases where the model correctly predicted the positive class.
2. **False Positive (FP):** These are the cases where the model incorrectly predicted the positive class (i.e., it predicted positive, but the actual class was negative).
3. **True Negative (TN):** These are the cases where the model correctly predicted the negative class.
4. **False Negative (FN):** These are the cases where the model incorrectly predicted the negative class (i.e., it predicted negative, but the actual class was positive).

![Confution Matrix](../images/confusion_matrix.png)

### Types of Classification

Confusion matrices are primarily used for binary classification problems (with two classes: positive and negative), but they can be extended to multiclass classification problems as well.

#### Example of a Binary Classification Confusion Matrix

|               | Predicted Positive | Predicted Negative |
|---------------|--------------------|--------------------|
| **Actual Positive** | 50 (TP)          | 10 (FN)           |
| **Actual Negative** | 5 (FP)           | 100 (TN)          |

- **True Positives (TP):** The model correctly predicted 50 cases as positive.
- **False Negatives (FN):** The model incorrectly predicted 10 positive cases as negative.
- **False Positives (FP):** The model incorrectly predicted 5 negative cases as positive.
- **True Negatives (TN):** The model correctly predicted 100 cases as negative.

![Binary Classification Confusion Matrix](../images/confusion_matrix_example.png)

### Evaluation Metrics Derived from the Confusion Matrix

From the confusion matrix, we can derive several important performance metrics that help in evaluating the model's effectiveness.

1. **Accuracy**: This is the ratio of the correctly predicted observations to the total observations.

   $$
   \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
   $$

2. **Precision (Positive Predictive Value)**: Precision is the ratio of correctly predicted positive observations to the total predicted positives. It answers the question: *Of all the positive predictions, how many were actually positive?*

   $$
   \text{Precision} = \frac{TP}{TP + FP}
   $$

3. **Recall (Sensitivity or True Positive Rate)**: Recall is the ratio of correctly predicted positive observations to all observations in the actual positive class. It answers the question: *Of all the actual positives, how many did we correctly identify?*

   $$
   \text{Recall} = \frac{TP}{TP + FN}
   $$

4. **F1-Score**: The F1-score is the weighted average of Precision and Recall. It is especially useful when the class distribution is imbalanced. A high F1-score means that both Precision and Recall are high.

   $$
   \text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
   $$

5. **Specificity (True Negative Rate)**: Specificity is the ratio of correctly predicted negative observations to all actual negative observations. It answers the question: *Of all the actual negatives, how many did we correctly identify?*

   $$
   \text{Specificity} = \frac{TN}{TN + FP}
   $$

6. **ROC Curve (Receiver Operating Characteristic Curve)**: This curve is a graphical plot that illustrates the diagnostic ability of a binary classifier. It plots the **True Positive Rate** (Recall) against the **False Positive Rate**.

7. **AUC (Area Under the Curve)**: The AUC is the area under the ROC curve. It provides a measure of how well the model can distinguish between classes. A higher AUC indicates a better model.

### Multiclass Confusion Matrix

While the confusion matrix is often discussed in the context of binary classification, it can be extended to multiclass problems (classification with more than two classes). In a multiclass confusion matrix, each class gets its own row and column, and the matrix size is increased accordingly.

For example, in a 3-class classification problem (classes A, B, and C), the confusion matrix might look like this:

|               | Predicted A | Predicted B | Predicted C |
|---------------|-------------|-------------|-------------|
| **Actual A**  | 30 (TP)     | 5 (FN)      | 2 (FN)      |
| **Actual B**  | 3 (FP)      | 25 (TP)     | 4 (FN)      |
| **Actual C**  | 1 (FP)      | 6 (FP)      | 28 (TP)     |

#### Key Metrics for Multiclass Problems

- **Precision, Recall, F1-Score**: These can be computed for each class individually and then averaged (using micro, macro, or weighted averages) to obtain a global metric.
- **Micro-average**: Counts the total true positives, false positives, false negatives across all classes, and computes the metrics.
- **Macro-average**: Computes the metrics for each class independently and then takes the average.
- **Weighted-average**: Similar to macro, but the contribution of each class is weighted by its support (the number of true instances for each class).

### How to Create a Confusion Matrix in Python

You can create a confusion matrix in Python using the `confusion_matrix` function from the `sklearn.metrics` module. Here’s an example:

In [13]:
from sklearn.metrics import confusion_matrix, classification_report

# True labels
y_true = [0, 1, 0, 1, 0, 1, 1, 0, 1, 1]

# Predicted labels
y_pred = [0, 1, 0, 0, 0, 1, 1, 0, 1, 1]

# Generate the confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[4 0]
 [1 5]]


In [15]:
cr = classification_report(y_true, y_pred)
print(cr)

              precision    recall  f1-score   support

           0       0.80      1.00      0.89         4
           1       1.00      0.83      0.91         6

    accuracy                           0.90        10
   macro avg       0.90      0.92      0.90        10
weighted avg       0.92      0.90      0.90        10



### Conclusion

The confusion matrix is a powerful tool to evaluate the performance of a classification model. It provides a clear picture of how well a model is performing, what types of errors it is making, and allows for the calculation of various performance metrics like accuracy, precision, recall, and F1-score.